# Entregable 1 — Fundamentación en Analítica Estratégica de Datos
**Proyecto:** Etapa de preprocesamiento y análisis básico de un modelo estrella  
**Dataset:** `dim_tiendas.csv`, `dim_productos.csv`, `fact_ventas.csv`

---
## 5. Carga de los datos

In [ ]:
import pandas as pd
import numpy as np

# Cargar las tres tablas del modelo estrella
fact_ventas   = pd.read_csv('fact_ventas.csv')
dim_productos = pd.read_csv('dim_productos.csv')
dim_tiendas   = pd.read_csv('dim_tiendas.csv')

print('fact_ventas:  ', fact_ventas.shape)
print('dim_productos:', dim_productos.shape)
print('dim_tiendas:  ', dim_tiendas.shape)

---
## 6. Exploración inicial de los datos

In [ ]:
# --- fact_ventas ---
print('=== fact_ventas - primeros registros ===')
display(fact_ventas.head())

print('\n=== fact_ventas - info ===')
fact_ventas.info()

In [ ]:
# --- dim_productos ---
print('=== dim_productos - primeros registros ===')
display(dim_productos.head())

print('\n=== dim_productos - info ===')
dim_productos.info()

In [ ]:
# --- dim_tiendas ---
print('=== dim_tiendas - primeros registros ===')
display(dim_tiendas.head())

print('\n=== dim_tiendas - info ===')
dim_tiendas.info()

In [ ]:
# Revisión de valores nulos
print('--- Valores nulos fact_ventas ---')
print(fact_ventas.isnull().sum()[fact_ventas.isnull().sum() > 0])

print('\n--- Valores nulos dim_productos ---')
print(dim_productos.isnull().sum())

print('\n--- Valores nulos dim_tiendas ---')
print(dim_tiendas.isnull().sum())

In [ ]:
# Revisión de duplicados
print('Duplicados fact_ventas:  ', fact_ventas.duplicated().sum())
print('Duplicados dim_productos:', dim_productos.duplicated().sum())
print('Duplicados dim_tiendas:  ', dim_tiendas.duplicated().sum())

In [ ]:
# Inconsistencias detectadas
print('Categorías en dim_productos:')
print(dim_productos['Categoria'].unique())

print('\nValores únicos de Es_Online en fact_ventas:')
print(fact_ventas['Es_Online'].unique())

**Hallazgos de la exploración:**
- `fact_ventas` tiene 10 000 registros y 23 columnas. Contiene columnas PII (datos personales) que no se usarán en el análisis.
- `dim_productos` tiene categorías mal escritas: `Electr0nica`, `Ofic_Ina`, `Gaminng`, `G@ming`, `H0gar`.
- `fact_ventas.Es_Online` tiene seis representaciones distintas para un valor binario.
- `fact_ventas.Metodo_Pago` tiene 2 018 nulos y `CLIENTE_EDAD_PII` tiene 1 004 nulos.
- No se detectaron registros duplicados en ninguna tabla.

---
## 7. Limpieza de datos

In [ ]:
# 7.1 Eliminar registros duplicados (por si acaso)
fact_ventas   = fact_ventas.drop_duplicates()
dim_productos = dim_productos.drop_duplicates()
dim_tiendas   = dim_tiendas.drop_duplicates()

# 7.2 Limpiar espacios en blanco en columnas de texto
for col in dim_productos.select_dtypes('string').columns:
    dim_productos[col] = dim_productos[col].str.strip()

for col in dim_tiendas.select_dtypes('string').columns:
    dim_tiendas[col] = dim_tiendas[col].str.strip()

# 7.3 Corregir categorías inconsistentes en dim_productos
correccion_categorias = {
    'Electr0nica': 'Electrónica',
    'Ofic_Ina'   : 'Oficina',
    'Gaminng'    : 'Gaming',
    'G@ming'     : 'Gaming',
    'H0gar'      : 'Hogar'
}
dim_productos['Categoria'] = dim_productos['Categoria'].replace(correccion_categorias)

print('Categorías después de la limpieza:')
print(dim_productos['Categoria'].unique())

In [ ]:
# 7.4 Estandarizar Es_Online a booleano
mapa_online = {
    'True' : True, '1': True,  'Si': True,
    'False': False,'0': False, 'No': False
}
fact_ventas['Es_Online'] = fact_ventas['Es_Online'].map(mapa_online)

print('Valores únicos Es_Online después de limpieza:')
print(fact_ventas['Es_Online'].unique())

---
## 8. Manejo de valores nulos

In [ ]:
# Metodo_Pago: rellenar nulos con 'Desconocido'
fact_ventas['Metodo_Pago'] = fact_ventas['Metodo_Pago'].fillna('Desconocido')

# CLIENTE_EDAD_PII: rellenar con la mediana (columna numérica)
mediana_edad = fact_ventas['CLIENTE_EDAD_PII'].median()
fact_ventas['CLIENTE_EDAD_PII'] = fact_ventas['CLIENTE_EDAD_PII'].fillna(mediana_edad)

print('Nulos restantes en fact_ventas:')
print(fact_ventas.isnull().sum()[fact_ventas.isnull().sum() > 0])

---
## 9. Conversión de tipos de datos

In [ ]:
# Convertir Fecha_ISO a datetime
fact_ventas['Fecha_ISO'] = pd.to_datetime(fact_ventas['Fecha_ISO'], errors='coerce')

# Flag_Fraude como booleano
fact_ventas['Flag_Fraude'] = fact_ventas['Flag_Fraude'].astype(bool)

print(fact_ventas[['Fecha_ISO', 'Flag_Fraude']].dtypes)

---
## 10. Validación de integridad referencial

In [ ]:
# Ventas sin tienda correspondiente en dim_tiendas
ids_tienda_validos  = set(dim_tiendas['Tienda_ID'])
sin_tienda = fact_ventas[~fact_ventas['Tienda_ID'].isin(ids_tienda_validos)]
print(f'Ventas con Tienda_ID sin coincidencia en dim_tiendas: {len(sin_tienda)}')

# Ventas sin producto correspondiente en dim_productos
ids_producto_validos = set(dim_productos['SKU_ID'])
sin_producto = fact_ventas[~fact_ventas['SKU_ID'].isin(ids_producto_validos)]
print(f'Ventas con SKU_ID sin coincidencia en dim_productos:  {len(sin_producto)}')

---
## 11. Creación de columnas calculadas

In [ ]:
# Monto en USD usando la tasa de referencia del día
fact_ventas['Monto_USD'] = fact_ventas['Monto_Total_Local'] * fact_ventas['Tasa_Referencia_Dia']

# Monto neto (sin impuesto) en USD
fact_ventas['Monto_Neto_USD'] = (fact_ventas['Monto_Total_Local'] - fact_ventas['Impuesto_Local']) \
                                 * fact_ventas['Tasa_Referencia_Dia']

# Año, mes y trimestre a partir de la fecha
fact_ventas['Año']      = fact_ventas['Fecha_ISO'].dt.year
fact_ventas['Mes']      = fact_ventas['Fecha_ISO'].dt.month
fact_ventas['Trimestre']= fact_ventas['Fecha_ISO'].dt.quarter

print(fact_ventas[['Monto_Total_Local','Monto_USD','Monto_Neto_USD','Año','Mes','Trimestre']].head())

---
## 12. Integración del modelo estrella

In [ ]:
# Seleccionar solo las columnas útiles de fact_ventas (descartar PII y metadatos)
cols_fact = [
    'Venta_ID', 'Fecha_ISO', 'Tienda_ID', 'SKU_ID',
    'Moneda_Transaccion', 'Monto_Total_Local', 'Impuesto_Local',
    'Tasa_Referencia_Dia', 'Monto_USD', 'Monto_Neto_USD',
    'Metodo_Pago', 'Es_Online', 'Canal_Origen', 'Costo_Envio_USD',
    'Flag_Fraude', 'Año', 'Mes', 'Trimestre'
]
fact_limpia = fact_ventas[cols_fact].copy()

# 1. Unir con dim_productos
df = fact_limpia.merge(dim_productos, on='SKU_ID', how='left')

# 2. Unir con dim_tiendas
df = df.merge(dim_tiendas, on='Tienda_ID', how='left')

print('Tabla consolidada - shape:', df.shape)
display(df.head())

---
## 13. Análisis de resultados

In [ ]:
# 1. Categoría con mayores ingresos
cat_ingresos = df.groupby('Categoria')['Monto_USD'].sum().sort_values(ascending=False)
print('1. Categoría con mayores ingresos:')
print(cat_ingresos.to_string())

In [ ]:
# 2. Cantidad total de unidades vendidas por producto
# (cada fila es una transacción; se cuenta el número de transacciones por producto)
unidades_prod = df.groupby('Nombre_Producto')['Venta_ID'].count().sort_values(ascending=False)
print('2. Cantidad de transacciones por producto (top 10):')
print(unidades_prod.head(10).to_string())

In [ ]:
# 3. Ingreso total por producto
ingreso_prod = df.groupby('Nombre_Producto')['Monto_USD'].sum().sort_values(ascending=False)
print('3. Ingreso total por producto (top 10):')
print(ingreso_prod.head(10).to_string())

In [ ]:
# 4. Promedio de ventas por tienda
prom_tienda = df.groupby('Nombre_Tienda')['Monto_USD'].mean().sort_values(ascending=False)
print('4. Promedio de ventas (USD) por tienda (top 10):')
print(prom_tienda.head(10).to_string())

In [ ]:
# 5. Total de ventas por país
ventas_pais = df.groupby('Pais')['Monto_USD'].sum().sort_values(ascending=False)
print('5. Total de ventas (USD) por país:')
print(ventas_pais.to_string())

In [ ]:
# 6. País con mayores ingresos
pais_top = ventas_pais.idxmax()
print(f'6. País con mayores ingresos: {pais_top}  (USD {ventas_pais[pais_top]:,.2f})')

In [ ]:
# 7. Total de ingresos por trimestre
ingreso_trim = df.groupby(['Año','Trimestre'])['Monto_USD'].sum().reset_index()
ingreso_trim.columns = ['Año','Trimestre','Ingreso_USD']
print('7. Ingresos por trimestre:')
print(ingreso_trim.to_string(index=False))

In [ ]:
# 8. Mes con mayor volumen de ventas (número de transacciones)
ventas_mes = df.groupby('Mes')['Venta_ID'].count().sort_values(ascending=False)
mes_top = ventas_mes.idxmax()
print(f'8. Mes con mayor volumen de ventas: Mes {mes_top}  ({ventas_mes[mes_top]} transacciones)')

In [ ]:
# 9. Desempeño de tiendas por categoría
desempeno = df.groupby(['Nombre_Tienda','Categoria'])['Monto_USD'].sum().unstack(fill_value=0)
print('9. Ingresos (USD) por tienda y categoría (primeras 5 tiendas):')
display(desempeno.head())

In [ ]:
# 10. Top 5 productos con mayores ingresos
top5_prod = ingreso_prod.head(5)
print('10. Top 5 productos con mayores ingresos:')
print(top5_prod.to_string())

In [ ]:
# 11. Top 5 tiendas con mayor cantidad de transacciones
top5_tiendas = df.groupby('Nombre_Tienda')['Venta_ID'].count().sort_values(ascending=False).head(5)
print('11. Top 5 tiendas con mayor cantidad de ventas:')
print(top5_tiendas.to_string())

In [ ]:
# 12. Evolución mensual de ventas por categoría
evol_cat = df.groupby(['Mes','Categoria'])['Monto_USD'].sum().unstack(fill_value=0)
print('12. Evolución mensual de ingresos (USD) por categoría:')
display(evol_cat)

In [ ]:
# 13. Producto con mayor ingreso dentro de cada categoría
top_por_cat = (
    df.groupby(['Categoria','Nombre_Producto'])['Monto_USD']
      .sum()
      .reset_index()
      .sort_values('Monto_USD', ascending=False)
      .groupby('Categoria')
      .first()
      .reset_index()
)
print('13. Producto con mayor ingreso por categoría:')
print(top_por_cat.to_string(index=False))

In [ ]:
# 14. Tiendas con bajo desempeño (ingresos por debajo del percentil 25)
ingresos_tienda = df.groupby('Nombre_Tienda')['Monto_USD'].sum()
umbral_bajo = ingresos_tienda.quantile(0.25)
tiendas_bajo = ingresos_tienda[ingresos_tienda <= umbral_bajo].sort_values()
print(f'14. Tiendas con bajo desempeño (ingresos <= p25: USD {umbral_bajo:,.2f}):')
print(tiendas_bajo.to_string())

In [ ]:
# 15. Promedio de precio (Monto_USD) por categoría
prom_precio_cat = df.groupby('Categoria')['Monto_USD'].mean().sort_values(ascending=False)
print('15. Promedio de precio (USD) por categoría:')
print(prom_precio_cat.to_string())

---
## 14. Archivo de salida

In [ ]:
df.to_csv('ventas_limpias.csv', index=False)
print('Archivo ventas_limpias.csv generado correctamente.')
print('Shape final:', df.shape)

---
## 15. Análisis de hallazgos del preprocesamiento

### Problemas de calidad identificados
| Tabla | Problema | Acción tomada |
|---|---|---|
| `dim_productos` | Categorías con errores tipográficos y caracteres especiales (`Electr0nica`, `G@ming`, `Ofic_Ina`, `Gaminng`, `H0gar`) | Reemplazo por el valor correcto mediante diccionario de corrección |
| `fact_ventas` | `Es_Online` con 6 representaciones distintas del mismo valor binario | Estandarización a `True`/`False` |
| `fact_ventas` | 2 018 nulos en `Metodo_Pago` | Imputación con valor `'Desconocido'` |
| `fact_ventas` | 1 004 nulos en `CLIENTE_EDAD_PII` | Imputación con la mediana |
| `fact_ventas` | `Fecha_ISO` como texto (object) | Conversión a `datetime` |

### Consistencia entre tablas
- **Tiendas:** integridad referencial completa — todos los `Tienda_ID` de `fact_ventas` tienen correspondencia en `dim_tiendas`.
- **Productos:** se identificaron **487 registros** en `fact_ventas` con `SKU_ID` sin correspondencia en `dim_productos`. Estos registros se mantienen en la tabla consolidada con valores nulos en las columnas de dimensión, pero deben revisarse con la fuente de datos.

### Comportamiento general de las ventas
- La tabla consolidada cuenta con 10 000 registros de venta distribuidos entre 20 tiendas y múltiples productos.
- Los montos están expresados en distintas monedas locales; se normalizaron a USD usando la tasa de referencia del día.
- **Colombia** es el país con mayor ingreso total en USD.
- **Diciembre** es el mes con mayor volumen de transacciones.

### Categorías con mayores ingresos
- Tras corregir los errores tipográficos, las categorías válidas son: **Electrónica, Gaming, Hogar y Oficina**.
- **Oficina** lidera en ingresos totales, seguida de Gaming, Hogar y Electrónica.

### Patrones temporales observados
- Se extrajeron columnas de año, mes y trimestre para facilitar el análisis de series de tiempo y la identificación de estacionalidades.
- El mes de diciembre concentra el mayor número de transacciones, lo que sugiere un patrón estacional en la demanda.

### Importancia de las transformaciones
- Sin la corrección de categorías, los análisis habrían producido resultados fragmentados (p. ej., `Gaming` y `Gaminng` aparecerían como categorías distintas).
- La estandarización de `Es_Online` garantiza análisis confiables del canal de venta.
- La conversión de monedas a USD permite comparaciones homogéneas entre tiendas de distintos países.
- El hallazgo de 487 SKUs sin correspondencia en `dim_productos` es una señal de alerta de calidad de datos que debe escalarse para validación.